In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import math
import time

# 1. Gerar dados de treino (números naturais de 1 a 10)
X_train = np.arange(1, 11, dtype=float)

# Fatorial real utilizando math.factorial
y_factorial = np.array([math.factorial(int(n)) for n in X_train], dtype=float)

# Aplicando a transformação proposta para suavização da curva combinatória
y_train_transformed = np.zeros_like(X_train)
for i, n in enumerate(X_train):
    fact = math.factorial(int(n))
    if fact == 1:
        y_train_transformed[i] = 1.0  # Mapeamento estipulado estável para n=1
    else:
        y_train_transformed[i] = 1.0 / np.log(fact)

print("X de Treino:", X_train)
print("Y Transformado (Alvo do PSO):", y_train_transformed)

In [ ]:
def tsk_inference(X, centers, sigmas, y_target=None, ridge_lambda=1e-2):
    """
    Motor TSK de Ordem 1 com Regularização Ridge Aumentada (Wiktorowicz, 2020).
    Permite matrizes de dados independentes do número de regras para generalização contínua.
    """
    # Garante que X seja um array numpy unidimensional ou bidimensional coerente
    X = np.atleast_1d(X)
    N = len(X)
    R = len(centers)
    
    # 1. Calcular ativação Gaussiana
    W = np.zeros((N, R))
    for i in range(N):
        for j in range(R):
            # Adicionado estabilizador numérico no denominador
            W[i, j] = np.exp(-((X[i] - centers[j])**2) / (2 * (sigmas[j]**2) + 1e-5))
            
    # Normalização segura das linhas
    row_sums = W.sum(axis=1, keepdims=True)
    W_norm = np.where(row_sums > 1e-12, W / row_sums, 1.0 / R)
    
    # 2. Matriz de design global
    X_hat = np.zeros((N, 2 * R))
    for j in range(R):
        X_hat[:, 2*j] = W_norm[:, j] * X
        X_hat[:, 2*j + 1] = W_norm[:, j]
        
    # 3. Estimação via Ridge com lambda maior para evitar MSE = 0 e Overfitting
    if y_target is not None:
        A = X_hat.T @ X_hat + ridge_lambda * np.eye(2 * R)
        try:
            P = np.linalg.inv(A) @ X_hat.T @ y_target
        except np.linalg.LinAlgError:
            P = np.linalg.pinv(A) @ X_hat.T @ y_target
        return X_hat @ P, P
    else:
        return X_hat

In [ ]:
def particle_swarm_optimization(X, y_trans, num_rules, pop_size=40, iterations=150, seed=42):
    np.random.seed(seed)
    
    # ESTRATÉGIA CHANG (2025): Reduzimos o número de regras e espalhamos os centros uniformemente
    # para forçar a interpolação contínua ao invés de decorar os pontos fixos.
    centers = np.linspace(X.min(), X.max(), num_rules)
    
    w = 0.6
    c1 = 1.8
    c2 = 1.8
    
    # Inicialização dos sigmas com valores maiores para cobrir o espaço entre os centros
    position = np.random.uniform(1.5, 4.0, size=(pop_size, num_rules))
    velocity = np.random.uniform(-0.2, 0.2, size=(pop_size, num_rules))
    
    pbest_position = position.copy()
    pbest_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in position])
    
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_position[gbest_idx].copy()
    
    global_best_fitness_history = []
    
    for it in range(iterations):
        for i in range(pop_size):
            r1 = np.random.rand(num_rules)
            r2 = np.random.rand(num_rules)
            
            velocity[i] = (w * velocity[i] + 
                           c1 * r1 * (pbest_position[i] - position[i]) + 
                           c2 * r2 * (gbest_position - position[i]))
            
            position[i] = position[i] + velocity[i]
            
            # CRUCIAL (Wiktorowicz, 2021): Impor um limite inferior estrito (ex: 1.0) para os sigmas.
            # Isso impede as gaussianas de murcharem e isolarem as regras no domínio contínuo.
            position[i] = np.clip(position[i], 1.0, 6.0)
            
            y_pred, _ = tsk_inference(X, centers, position[i], y_trans)
            current_mse = np.mean((y_trans - y_pred)**2)
            
            if current_mse < pbest_fitness[i]:
                pbest_fitness[i] = current_mse
                pbest_position[i] = position[i].copy()
                
                if current_mse < pbest_fitness[gbest_idx]:
                    gbest_position = position[i].copy()
                    gbest_idx = i
                    
        global_best_fitness_history.append(pbest_fitness[gbest_idx])
        
    return gbest_position, global_best_fitness_history, centers

In [ ]:
seeds = [10, 42, 100, 2026, 999]
results = []
convergence_curves = []

print("Iniciando o Protocolo Experimental de Generalização (5 Sementes)... \n")
start_time = time.time()

# Definimos 4 regras ao invés de 10. Menos regras = maior compartilhamento fuzzy (Chang, 2025).
NUM_REGRAS_REDUZIDO = 4 

for s in seeds:
    best_sigmas, history, computed_centers = particle_swarm_optimization(
        X_train, y_train_transformed, num_rules=NUM_REGRAS_REDUZIDO, seed=s
    )
    results.append({
        'seed': s,
        'best_sigmas': best_sigmas,
        'centers': computed_centers,
        'final_mse': history[-1]
    })
    convergence_curves.append(history)
    print(f"Seed {s:4d} finalizada. MSE real controlado: {history[-1]:.6f}")

total_time = time.time() - start_time
mses = [r['final_mse'] for r in results]

print("\n--- MATRIZ DE MÉTRICAS REAIS CORRIGIDAS ---")
print(f"Melhor MSE: {np.min(mses):.6f}")
print(f"Desvio Padrão: {np.std(mses):.6f}")

In [ ]:
# Resgatando a melhor execução estável
best_run = results[np.argmin(mses)]
optimal_sigmas = best_run['best_sigmas']
centers_fixed = best_run['centers']

# Ajustando o modelo ótimo com os dados de treino
y_pred_trans, optimal_P = tsk_inference(X_train, centers_fixed, optimal_sigmas, y_train_transformed)

# Domínio contínuo estendido para verificar se a aproximação curvou corretamente
X_continuous = np.linspace(1.0, 10.0, 300)
X_hat_continuous = tsk_inference(X_continuous, centers_fixed, optimal_sigmas)
y_pred_continuous_trans = X_hat_continuous @ optimal_P

# Inversão matemática rigorosa
y_pred_factorial_continuous = np.exp(1.0 / y_pred_continuous_trans)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for i, hist in enumerate(convergence_curves):
    plt.plot(hist, label=f"Seed {seeds[i]}")
plt.title("Curvas de Convergência (Interpolação Controlada)")
plt.xlabel("Iterações")
plt.ylabel("MSE Real do Domínio")
plt.yscale('log')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(X_continuous, gamma(X_continuous + 1), 'g-', label='Função Gamma Analítica $\Gamma(x+1)$', alpha=0.7)
plt.plot(X_continuous, y_pred_factorial_continuous, 'r--', label='Aproximação Suavizada TSK + PSO', linewidth=2)
plt.scatter(X_train, y_factorial, color='black', zorder=5, label='Fatoriais Reais ($n!$)')
plt.title("Aproximação Contínua Corrigida do Fatorial")
plt.xlabel("Entrada ($x$)")
plt.ylabel("Valor (Escala Logarítmica)")
plt.yscale('log')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()